In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional

### Label Smoothing Focal Loss (Rekomendasi Utama)

In [3]:
class LabelSmoothingFocalLoss(nn.Module):
    def __init__(
        self,
        num_classes: int = 6,
        smoothing: float = 0.1,
        gamma: float = 2.0,
        alpha: Optional[torch.Tensor] = None,
        reduction: str = "mean"
    ):
        super().__init__()
        self.num_classes = num_classes
        self.smoothing = smoothing
        self.gamma = gamma
        self.alpha = alpha
        self.reduction = reduction

        self.eps = 1e-8
 
    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        B, C = logits.shape
        device = logits.device
 
        # ── Buat soft target dengan label smoothing ──
        smooth_val = self.smoothing / (C - 1)
        soft_targets = torch.full((B, C), smooth_val, device=device)
        soft_targets.scatter_(1, targets.unsqueeze(1), 1.0 - self.smoothing)
 
        # ── Hitung log probabilities ──
        log_prob = F.log_softmax(logits, dim=1)  # (B, C)
        prob     = torch.exp(log_prob)            # (B, C)
 
        # ── Focal weight: (1 - p_t)^gamma ──
        # p_t adalah probability untuk kelas yang benar
        p_t = prob.gather(1, targets.unsqueeze(1)).squeeze(1)  # (B,)
        focal_weight = (1.0 - p_t + self.eps) ** self.gamma    # (B,)
 
        # ── Cross entropy dengan soft targets ──
        loss_per_sample = -(soft_targets * log_prob).sum(dim=1)  # (B,)
 
        # ── Terapkan focal weight ──
        loss_per_sample = focal_weight * loss_per_sample
 
        # ── Terapkan class alpha weight (opsional) ──
        if self.alpha is not None:
            alpha_t = self.alpha.to(device)[targets]  # (B,)
            loss_per_sample = alpha_t * loss_per_sample
 
        # ── Reduction ──
        if self.reduction == "mean":
            return loss_per_sample.mean()
        elif self.reduction == "sum":
            return loss_per_sample.sum()
        else:
            return loss_per_sample


### Mixup Criterion

In [4]:
class MixupCriterion(nn.Module): 
    def __init__(self, base_criterion: nn.Module):
        super().__init__()
        self.base = base_criterion
 
    def forward(
        self,
        logits: torch.Tensor,
        y_a: torch.Tensor,
        y_b: torch.Tensor,
        lam: float
    ) -> torch.Tensor:
        return lam * self.base(logits, y_a) + (1 - lam) * self.base(logits, y_b)

### Class-Weighted Focal Loss (alternatif)

In [5]:
class ClassWeightedFocalLoss(nn.Module): 
    def __init__(
        self,
        class_weights: torch.Tensor,
        gamma: float = 2.0,
    ):
        super().__init__()
        self.register_buffer("class_weights", class_weights)
        self.gamma = gamma
 
    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        log_prob = F.log_softmax(logits, dim=1)
        prob     = torch.exp(log_prob)
 
        # Ambil prob dan log_prob untuk kelas target
        p_t   = prob.gather(1, targets.unsqueeze(1)).squeeze(1)
        lp_t  = log_prob.gather(1, targets.unsqueeze(1)).squeeze(1)
 
        # Focal weight
        focal_w = (1.0 - p_t) ** self.gamma
 
        # Class weight
        class_w = self.class_weights[targets]
 
        loss = -class_w * focal_w * lp_t
        return loss.mean()

### Helper: Hitung class weights dari dataset

In [6]:
def compute_class_weights(labels: list, num_classes: int = 6) -> torch.Tensor:
    import numpy as np
    labels_arr = np.array(labels)
    counts = np.bincount(labels_arr, minlength=num_classes).astype(float)
    counts = np.maximum(counts, 1)  
    weights = 1.0 / counts
    weights = weights / weights.sum() * num_classes  
    return torch.tensor(weights, dtype=torch.float32)

### Konfigurasi yang Direkomendasikan

In [7]:
def get_recommended_criterion(
    use_class_weights: bool = False,
    class_weights: Optional[torch.Tensor] = None,
    device: str = "cuda"
) -> nn.Module:
    if use_class_weights and class_weights is not None:
        alpha = class_weights.to(device)
    else:
        alpha = None
 
    criterion = LabelSmoothingFocalLoss(
        num_classes=6,
        smoothing=0.1,    
        gamma=2.0,        
        alpha=alpha,
        reduction="mean"
    ).to(device)
 
    return criterion

### Mixup helper function (untuk dipakai di training loop)

In [8]:
def mixup_data(x: torch.Tensor, y: torch.Tensor, alpha: float = 0.4):
    import numpy as np
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1.0
 
    B = x.size(0)
    index = torch.randperm(B, device=x.device)
 
    x_mixed = lam * x + (1 - lam) * x[index]
    y_a = y
    y_b = y[index]
 
    return x_mixed, y_a, y_b, lam

### Contoh penggunaan di training loop

In [9]:
TRAINING_LOOP_EXAMPLE = """
# ── Di awal training ──
from loss_functions import (
    get_recommended_criterion, MixupCriterion, mixup_data
)
 
base_criterion = get_recommended_criterion(device=device)
mixup_criterion = MixupCriterion(base_criterion)
USE_MIXUP = True  # Toggle mixup on/off
 
# ── Di dalam training loop ──
for images, labels in train_loader:
    images, labels = images.to(device), labels.to(device)
    optimizer.zero_grad()
 
    with torch.amp.autocast("cuda"):
        if USE_MIXUP and random.random() < 0.5:
            images_mix, y_a, y_b, lam = mixup_data(images, labels, alpha=0.4)
            logits = model(images_mix)
            loss = mixup_criterion(logits, y_a, y_b, lam)
        else:
            logits = model(images)
            loss = base_criterion(logits, labels)
 
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()
"""
 
if __name__ == "__main__":
    # Sanity check
    B, C = 8, 6
    logits  = torch.randn(B, C)
    targets = torch.randint(0, C, (B,))
 
    crit = LabelSmoothingFocalLoss(num_classes=C, smoothing=0.1, gamma=2.0)
    loss = crit(logits, targets)
    print(f"LabelSmoothingFocalLoss: {loss.item():.4f}")
 
    weights = compute_class_weights(targets.tolist(), num_classes=C)
    print(f"Class weights: {weights}")
 
    x = torch.randn(B, 3, 224, 224)
    x_mix, ya, yb, lam = mixup_data(x, targets)
    print(f"Mixup lambda: {lam:.4f}")
    print("loss_functions.py OK")

LabelSmoothingFocalLoss: 1.4332
Class weights: tensor([0.6207, 0.4138, 1.2414, 1.2414, 1.2414, 1.2414])
Mixup lambda: 0.7873
loss_functions.py OK
